In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kazanova/sentiment140")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sentiment140' dataset.
Path to dataset files: /kaggle/input/sentiment140


In [2]:
import numpy as np
import pandas as pd

# NLP Libraries
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML Libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [3]:
# Download required resources
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
# Example: CSV with 'review' and 'sentiment'
column_names = ['target', 'ids', 'date', 'flag', 'user', 'text']
df = pd.read_csv('/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv', encoding='latin1', header=None, names=column_names)

df.head()

,target,ids,date,flag,user,text
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [5]:
print("Shape:", df.shape)

print("\nClass Distribution:\n", df['target'].value_counts())

print("\nSample Tweet:\n", df['text'][0])

Shape: (1600000, 6)

Class Distribution:
 target
0    800000
4    800000
Name: count, dtype: int64

Sample Tweet:
 @switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D


In [6]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_tweet(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove mentions (@user)
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags (#tag → tag)
    text = re.sub(r'#', '', text)

    # Remove emojis & special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = text.split()

    # Remove stopwords + Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]

    return " ".join(tokens)

In [7]:
df['clean_tweet'] = df['text'].apply(preprocess_tweet)

df[['text', 'clean_tweet']].head()

,text,clean_tweet
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",thats bummer shoulda got david carr third day
1,is upset that he can't update his Facebook by ...,upset cant update facebook texting might cry r...
2,@Kenichan I dived many times for the ball. Man...,dived many time ball managed save rest go bound
3,my whole body feels itchy and like its on fire,whole body feel itchy like fire
4,"@nationwideclass no, it's not behaving at all....",behaving im mad cant see


In [8]:
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['clean_tweet'])

In [9]:
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['clean_tweet'])

In [10]:
y = df['target']

In [11]:
X_train_bow, X_test_bow, y_train, y_test = train_test_split(X_bow, y, test_size=0.2, random_state=42)

X_train_tfidf, X_test_tfidf, _, _ = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

In [12]:
lr = LogisticRegression(max_iter=200)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

In [13]:
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)

y_pred_nb = nb.predict(X_test_bow)

In [14]:
def evaluate(y_test, y_pred, model_name):

    print(f"\n📊 {model_name}")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall   :", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score :", f1_score(y_test, y_pred, average='weighted'))

    print("\nClassification Report:\n")
    print(classification_report(y_test, y_pred))

In [16]:
evaluate(y_test, y_pred_lr, "Logistic Regression (TF-IDF)")
evaluate(y_test, y_pred_nb, "Naive Bayes (BoW)")


📊 Logistic Regression (TF-IDF)
Accuracy : 0.77323125
Precision: 0.7736859880608312
Recall   : 0.77323125
F1 Score : 0.7731156684803012

Classification Report:

              precision    recall  f1-score   support

           0       0.78      0.75      0.77    159494
           4       0.76      0.79      0.78    160506

    accuracy                           0.77    320000
   macro avg       0.77      0.77      0.77    320000
weighted avg       0.77      0.77      0.77    320000


📊 Naive Bayes (BoW)
Accuracy : 0.76081875
Precision: 0.7609753006947271
Recall   : 0.76081875
F1 Score : 0.7607956073361365

Classification Report:

              precision    recall  f1-score   support

           0       0.75      0.77      0.76    159494
           4       0.77      0.75      0.76    160506

    accuracy                           0.76    320000
   macro avg       0.76      0.76      0.76    320000
weighted avg       0.76      0.76      0.76    320000



In [17]:
def predict_sentiment(text, vectorizer, model):

    cleaned = preprocess_tweet(text)
    vectorized = vectorizer.transform([cleaned])

    prediction = model.predict(vectorized)

    return prediction[0]

In [18]:
sample = "I love this product! Amazing experience 😍"

print("Tweet:", sample)
print("Predicted Sentiment:", predict_sentiment(sample, tfidf, lr))

Tweet: I love this product! Amazing experience 😍
Predicted Sentiment: 4
